In [1]:
from dexter.config.constants import Split
from dexter.data.loaders.RetrieverDataset import RetrieverDataset
from dexter.utils.metrics.ExactMatch import ExactMatch
from dexter.retriever.dense.Contriever import Contriever
from dexter.utils.metrics.SimilarityMatch import CosineSimilarity
from dexter.utils.metrics.retrieval.RetrievalMetrics import RetrievalMetrics
from dexter.llms.flant5_engine import FlanT5Engine
import pandas as pd
import joblib

In [2]:
import configparser

config = configparser.ConfigParser()
config['Data-Path'] = {
    'wikimultihopqa' : 'data/',
    'wikimultihopqa-corpus' : 'wiki_musique_corpus.json'
}

with open('config.ini', 'w') as configfile:
    config.write(configfile)

print("config.ini created successfully!")

config.ini created successfully!


In [3]:
loader = RetrieverDataset("wikimultihopqa", "wikimultihopqa-corpus", "config.ini", Split.DEV, tokenizer=None)

Transforming passage dataset: 100%|██████████| 563424/563424 [00:01<00:00, 437745.47it/s]


Harley-Davidson Harley-Davidson
KeysView(<Section: Data-Path>)
12576


100%|██████████| 1200/1200 [02:51<00:00,  7.00it/s]


In [4]:
joblib.dump(loader, "loader.pkl")

['loader.pkl']

In [5]:
loader = joblib.load("loader.pkl")

In [6]:
from dexter.data.datastructures.hyperparameters.dpr import DenseHyperParams

In [7]:
config_instance = DenseHyperParams(query_encoder_path="facebook/contriever", document_encoder_path="facebook/contriever", batch_size=32, show_progress_bar=True)

In [8]:
queries, qrels, corpus = loader.qrels()

In [9]:
contrvr_search = Contriever(config_instance)   
similarity_measure = CosineSimilarity()

C:\Users\San\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\San\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\modeling_utils.py:463: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the 

In [8]:
response = {}

In [ ]:
k_values = [1, 3, 5]
for k in k_values:
    response[k] = contrvr_search.retrieve(corpus, queries, k, similarity_measure, chunk=True, chunksize=40)

In [10]:
joblib.dump(response, "response.pkl")

['response.pkl']

In [10]:
response = joblib.load("response.pkl")

In [11]:
metric = ExactMatch()

In [12]:
flant5 = FlanT5Engine(response)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
joblib.dump(flant5, "flant5_response.pkl")

['flant5_response.pkl']

In [14]:
flant5 = joblib.load("flant5_response.pkl")

In [41]:
corpus[496418].text()

"2018 Cannes Film Festival Official poster of the 71st Cannes Film Festival featuring Jean - Paul Belmondo and Anna Karina in Pierrot le Fou (1965) Opening film Everybody Knows Closing film The Man Who Killed Don Quixote Location Cannes, France Founded 1946 Awards Palme d'Or (Shoplifters) Hosted by Édouard Baer No. of films 21 (In Competition) 18 (Un Certain Regard) Festival date 8 -- 19 May 2018 Website festival-cannes.com/en"

### Stuff because stuff is weird

qrels gives us relevant documents for each query indexed by query id

responses gives us top-k relevant documents for each query acc to contriever indexed by query id

queries is just the list of questions, use q.text() to get string value, q.id() to get query id

corpus has evidences indexed by the id present in qrels or responses id


In [19]:
metrics = RetrievalMetrics(k_values=[1, 3, 5])
print(metrics.evaluate_retrieval(qrels=qrels, results=response[5]))

({'NDCG@1': 0.45083, 'NDCG@3': 0.34517, 'NDCG@5': 0.28381}, {'MAP@1': 0.04535, 'MAP@3': 0.07725, 'MAP@5': 0.08954}, {'Recall@1': 0.04535, 'Recall@3': 0.09471, 'Recall@5': 0.11961}, {'P@1': 0.45083, 'P@3': 0.31389, 'P@5': 0.23783})


In [20]:
raw_data = loader.base_dataset.raw_data

In [68]:
for r in raw_data:
    print(r.question.text())

Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
Are director of film Move (1970 Film)

In [52]:
response[5]

{'13f5ad2c088c11ebbd6fac1f6bf848b6': {'252094': 0.46607813239097595,
  '98709': 0.4693426787853241,
  '59201': 0.4693426787853241,
  '496418': 0.479369193315506,
  '72056': 0.47573965787887573},
 '3057c6c4086111ebbd5dac1f6bf848b6': {'225310': 0.4121391475200653,
  '411824': 0.41234707832336426,
  '205019': 0.42697057127952576,
  '226174': 0.4234209656715393,
  '187876': 0.4499715268611908},
 '89bc944808a111ebbd79ac1f6bf848b6': {'31309': 0.4021065831184387,
  '94595': 0.4167948365211487,
  '357169': 0.40240007638931274,
  '433265': 0.4683046042919159,
  '297135': 0.46830445528030396},
 '633f80660bdd11eba7f7acde48001122': {'123983': 0.4594964385032654,
  '118785': 0.4687654376029968,
  '193327': 0.569388210773468,
  '503936': 0.48808377981185913,
  '211848': 0.47718513011932373},
 '2dc3f9740bda11eba7f7acde48001122': {'104342': 0.45861488580703735,
  '152580': 0.4632769525051117,
  '80533': 0.4811129570007324,
  '475972': 0.47994759678840637,
  '518687': 0.4840882420539856},
 '61df8a820bd

In [21]:
evidences = {}
for q in response[5].keys():
    evidence_ids = list(response[5][q].keys())
    evidence_string_list = []
    for id in evidence_ids:
        evidence_string_list.append(corpus[int(id)].text())
    evidences[q] = evidence_string_list

print(evidences)

{'8813f87c0bdd11eba7f7acde48001122': ['The Last Family  is a 2016 Polish biographical film directed by Jan P. Matuszyński. The film won the Golden Lions for best film at the 2016 Gdynia Film Festival.', 'The Last Family is a 2016 Polish biographical film directed by Jan P. Matuszyński. The film won the Golden Lions for best film at the 2016 Gdynia Film Festival.', 'Polish- Russian War( Wojna polsko- ruska) is a 2009 Polish film directed by Xawery Żuławski based on the novel Polish- Russian War under the white- red flag by Dorota Masłowska.', 'Polish-Russian War (Wojna polsko-ruska) is a 2009 Polish film directed by Xawery Żuławski based on the novel Polish-Russian War under the white-red flag by Dorota Masłowska.', 'Sitora Shokhinovna Alieva – film expert, director of the IFF “ Faces of love ” and the IIF Sochi, artistic director of the largest Russian national film festival “ Kinotavr ”, Russian Ministry of Culture film expert, lecturer at film schools and universities, juror at numer

In [22]:
len(raw_data)

12000

In [39]:
r_samples = []
len_q = 0
for row in raw_data:
    if row.question.id() in r_samples:
        continue
    else:
        r_samples.append(row)
        len_q += 1
        if len_q == 100:
            break

In [41]:
question_df = {"questions":[], "answers":[]}
system_prompt = "Given the question and context, think step by step and output final answer for the question using information in the context and give answer in form of [Final Answer]: \n"
matches = 0
mismatches = 0
ids = []
count = 0
for row in r_samples:
    # if row.question.id() in ids:
    #     continue
    # else:
    #     ids.append(row.question.id())
    top_5 = " ".join(evidences[row.question.id()][0:5])
    # print(top_3)
    user_prompt = "[Question]: When does monsoon season end in the state the area code 575 is located? \
[Final Answer]: mid-September. \
[Question]: What is the current official currency in the country where Ineabelle Diaz is a citizen? \
[Final Answer]: United States dollar. \
[Question]: Where was the person who founded the American Institute of Public Opinion in 1935 born? \
[Final Answer]: Jefferson. \
[Question]: What language is used by the director of Tiffany Memorandum? \
[Final Answer]: Italian. \
[Question]: What is the sports team the person played for who scored the first touchdown in Superbowl 1? \
[Final Answer]: Green Bay Packers. \
[Question]: The birth country of Jayantha Ketagoda left the British Empire when? \
[Final Answer]: February 4, 1948.\n\n Follow the above example and given the evidence, evidence: " + top_5 + " \n use this information, think step by step and answer the question:" + row.question.text()
    # print(user_prompt)
    chain_answer = flant5.get_flant5_completion(system_prompt + " " + user_prompt)
    if "not possible" in chain_answer.lower():
        mismatches+=1
        continue
    elif "unknown" in chain_answer.lower():
        mismatches+=1
        continue
    elif "?" in chain_answer.lower():
        mismatches+=1
        continue
    elif "no answer" in chain_answer.lower():
        mismatches+=1
        continue
    elif len(chain_answer.split("[Final Answer]:")) >1:
        answer = chain_answer.split("[Final Answer]:")[-1]
        print("************",answer,row.answer.text())
        if row.answer.text().lower() in answer.lower():
            matches+=1
        else:
            mismatches+=1
    else:
        mismatches+=1
    question_df["answers"].append(chain_answer)
    question_df["questions"].append(row.question.text())
        
final_questions = pd.DataFrame(question_df)
EM = matches/(matches+mismatches)
print("EM: ", EM)
final_questions.to_csv("flant5_zero_shot_top5.tsv", sep="\t", index=False)
        

************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  Sitora Shokhinovna Alieva Małgorzata Braunek
************  The Mask of Fu Manchu The Mask Of Fu Manchu
************  The Mask of Fu Manchu The Mask Of Fu Manchu
************  The Mask of Fu Manchu The Mask Of Fu Manchu
************  The Mask of Fu Manchu The Mask Of Fu Manchu
************  The Mask of Fu Manchu The Mask Of Fu Manchu
************  The Mask of Fu Manchu The Mask Of Fu Manchu
************  The Mask of Fu Manchu The Mask Of Fu Manchu
****

In [42]:
question_df = {"questions":[], "answers":[]}
system_prompt = "Given the question and context, think step by step and output final answer for the question using information in the context and give answer in form of [Final Answer]: \n"
matches = 0
mismatches = 0
ids = []
count = 0
for row in r_samples:
    # if row.question.id() in ids:
    #     continue
    # else:
    #     ids.append(row.question.id())
    top_3 = " ".join(evidences[row.question.id()][0:3])
    # print(top_3)
    user_prompt = "[Question]: When does monsoon season end in the state the area code 575 is located? \
[Final Answer]: mid-September. \
[Question]: What is the current official currency in the country where Ineabelle Diaz is a citizen? \
[Final Answer]: United States dollar. \
[Question]: Where was the person who founded the American Institute of Public Opinion in 1935 born? \
[Final Answer]: Jefferson. \
[Question]: What language is used by the director of Tiffany Memorandum? \
[Final Answer]: Italian. \
[Question]: What is the sports team the person played for who scored the first touchdown in Superbowl 1? \
[Final Answer]: Green Bay Packers. \
[Question]: The birth country of Jayantha Ketagoda left the British Empire when? \
[Final Answer]: February 4, 1948.\n\n Follow the above example and given the evidence, evidence: " + top_3 + " \n use this information, think step by step and answer the question:" + row.question.text()
    # print(user_prompt)
    chain_answer = flant5.get_flant5_completion(system_prompt + " " + user_prompt)
    if "not possible" in chain_answer.lower():
        mismatches+=1
        continue
    elif "unknown" in chain_answer.lower():
        mismatches+=1
        continue
    elif "?" in chain_answer.lower():
        mismatches+=1
        continue
    elif "no answer" in chain_answer.lower():
        mismatches+=1
        continue
    elif len(chain_answer.split("[Final Answer]:")) >1:
        answer = chain_answer.split("[Final Answer]:")[-1]
        print("************",answer,row.answer.text())
        if row.answer.text().lower() in answer.lower():
            matches+=1
        else:
            mismatches+=1
    else:
        mismatches+=1
    question_df["answers"].append(chain_answer)
    question_df["questions"].append(row.question.text())
        
final_questions = pd.DataFrame(question_df)
EM = matches/(matches+mismatches)
print("EM: ", EM)
final_questions.to_csv("flant5_zero_shot_top3.tsv", sep="\t", index=False)

************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  Dorota Masowska Małgorzata Braunek
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask O

In [43]:
question_df = {"questions":[], "answers":[]}
system_prompt = "Given the question and context, think step by step and output final answer for the question using information in the context and give answer in form of [Final Answer]: \n"
matches = 0
mismatches = 0
ids = []
count = 0
for row in r_samples:
    # if row.question.id() in ids:
    #     continue
    # else:
    #     ids.append(row.question.id())
    top_1 = " ".join(evidences[row.question.id()][0:1])
    # print(top_3)
    user_prompt = "[Question]: When does monsoon season end in the state the area code 575 is located? \
[Final Answer]: mid-September. \
[Question]: What is the current official currency in the country where Ineabelle Diaz is a citizen? \
[Final Answer]: United States dollar. \
[Question]: Where was the person who founded the American Institute of Public Opinion in 1935 born? \
[Final Answer]: Jefferson. \
[Question]: What language is used by the director of Tiffany Memorandum? \
[Final Answer]: Italian. \
[Question]: What is the sports team the person played for who scored the first touchdown in Superbowl 1? \
[Final Answer]: Green Bay Packers. \
[Question]: The birth country of Jayantha Ketagoda left the British Empire when? \
[Final Answer]: February 4, 1948.\n\n Follow the above example and given the evidence, evidence: " + top_1 + " \n use this information, think step by step and answer the question:" + row.question.text()
    # print(user_prompt)
    chain_answer = flant5.get_flant5_completion(system_prompt + " " + user_prompt)
    if "not possible" in chain_answer.lower():
        mismatches+=1
        continue
    elif "unknown" in chain_answer.lower():
        mismatches+=1
        continue
    elif "?" in chain_answer.lower():
        mismatches+=1
        continue
    elif "no answer" in chain_answer.lower():
        mismatches+=1
        continue
    elif len(chain_answer.split("[Final Answer]:")) >1:
        answer = chain_answer.split("[Final Answer]:")[-1]
        print("************",answer,row.answer.text())
        if row.answer.text().lower() in answer.lower():
            matches+=1
        else:
            mismatches+=1
    else:
        mismatches+=1
    question_df["answers"].append(chain_answer)
    question_df["questions"].append(row.question.text())
        
final_questions = pd.DataFrame(question_df)
EM = matches/(matches+mismatches)
print("EM: ", EM)
final_questions.to_csv("flant5_zero_shot_top1.tsv", sep="\t", index=False)

************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  Anna Matuszyska Małgorzata Braunek
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask Of Fu Manchu
************  The Mask Of Fu Manchu The Mask O

KeyboardInterrupt: 